# AlphaFold Ensemble Competition Screen
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Drew-Thomson/AFcompetition/blob/main/AF_Competition_Screen.ipynb)

This notebook automates the process of generating AlphaFold predictions for a target protein and two potential ligands to determine which ligand preferentially occupies the user-specified binding site.

Requirements: `colabfold_batch` must be installed and accessible in the system path.

In [ ]:
# @title 1. Setup Environment
# @markdown Run this cell to install ColabFold and download necessary files.
import os
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    try:
        subprocess.check_output('nvidia-smi')
        print("GPU detected. Proceeding...")
    except Exception:
        print("WARNING: No GPU detected! Please ensure you are using a GPU runtime BEFORE proceeding (Runtime -> Change runtime type -> T4 GPU).")
    print("Installing dependencies (this may take a few minutes)...")
    !pip install -q -U biopython py3Dmol "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"
    if not os.path.exists("af_competition.py"):
        print("Downloading af_competition.py...")
        !wget -q https://raw.githubusercontent.com/Drew-Thomson/AFcompetition/main/af_competition.py
    print("Setup complete!")
else:
    print("Running locally. Dependencies assumed to be met.")

In [ ]:
# @title 2. (Optional) Mount Google Drive for persistent storage
# @markdown Run this to save your results permanently to your Google Drive.
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_OUTPUT_DIRECTORY = '/content/drive/MyDrive/AF_Competition_Results'
    os.makedirs(BASE_OUTPUT_DIRECTORY, exist_ok=True)
    print(f"Results will be saved to {BASE_OUTPUT_DIRECTORY}")
else:
    BASE_OUTPUT_DIRECTORY = './colabfold_results'

In [ ]:
import os
import py3Dmol
import matplotlib.pyplot as plt
from af_competition import run_colabfold, process_ensemble, optimize_threshold  # noqa: F401


In [ ]:
# @title Configuration
# @markdown Define the sequences, binding site residues, and runtime parameters.

# Sequences (Amino Acids)
TARGET_SEQ = "QETLVRPKPLLLKLLKSVGAQKDTYTMKEVLFYLGQYIMTKRLYDAAQQHIVYCSNDLLGDLFGVPSFSVKEHRKIYTMIYRNLVVVN" # @param {type:"string"}
LIGAND_1_SEQ = "ETFSDLWKLLPE" # @param {type:"string"}
LIGAND_2_SEQ = "LTFEHYWAQLTS" # @param {type:"string"}

# Binding site definition (Residue indices on the Target chain, 1-indexed)
BINDING_SITE = "38, 76" # @param {type:"string"}
BINDING_SITE_RESIDUES = [int(x.strip()) for x in BINDING_SITE.split(',') if x.strip()]

# Execution parameters
NUM_SEEDS = 20 # @param {type:"integer"}
RUN_NAME = "competition" # @param {type:"string"}
MIN_PLDDT = 80.0 # @param {type:"number"}

# BASE_OUTPUT_DIRECTORY is set in the Google Drive cell above.
# If running locally or without Drive, default to local folder:
if 'BASE_OUTPUT_DIRECTORY' not in locals():
    BASE_OUTPUT_DIRECTORY = "./colabfold_results"


In [ ]:
# @title 2. Execution
# @markdown Generate the models using `colabfold_batch`. *Note: This step may take significant time depending on sequence length and hardware.*

# Execute ColabFold. Uncomment to run.
ACTUAL_OUTPUT_DIR = run_colabfold(TARGET_SEQ, LIGAND_1_SEQ, LIGAND_2_SEQ, BASE_OUTPUT_DIRECTORY, run_name=RUN_NAME, num_seeds=NUM_SEEDS)
print(f"Models saved to: {ACTUAL_OUTPUT_DIR}")

# If you are skipping execution to analyze a previous run, manually define the directory here:
# ACTUAL_OUTPUT_DIR = "./colabfold_results/competition"
print("ColabFold execution block parsed.")


In [ ]:
# @title 3. Analysis
# @markdown Parse the resulting PDB files to calculate interface heavy-atom distances between the target binding site and the ligands.

# Process the PDB files (extracting distances and pLDDT)
all_results = process_ensemble(ACTUAL_OUTPUT_DIR, BINDING_SITE_RESIDUES, LIGAND_1_SEQ, LIGAND_2_SEQ, distance_threshold=0.0)
print(f"Total models parsed: {len(all_results)}")

# Filter by pLDDT
results = [r for r in all_results if r["mean_plddt"] >= MIN_PLDDT]
failed = len(all_results) - len(results)
print(f"Models passing pLDDT threshold ({MIN_PLDDT}): {len(results)}")
print(f"Models failing pLDDT threshold: {failed}")

if len(results) == 0:
    print("No models passed the pLDDT threshold.")
else:
    # Optimize the distance threshold
    optimization_result = optimize_threshold(results, min_thresh=2.0, max_thresh=8.0, step=0.25)
    optimal_threshold = optimization_result['optimal_threshold']
    stats = optimization_result['stats']
    
    print(f"\nOptimal distance threshold found at {optimal_threshold} Å\n")

    def print_files(label, count, files):
        print(f"{label}: {count}")
        if files:
            print("  Files:")
            for f in files:
                print(f"    - {f}")
        else:
            print("  Files: None")
    
    print_files("Ligand 1 exclusively bound", stats['lig1_wins'], stats['lig1_files'])
    print_files("Ligand 2 exclusively bound", stats['lig2_wins'], stats['lig2_files'])
    print_files("Both bound", stats['both_bound'], stats['both_files'])
    print_files("Neither bound", stats['neither_bound'], stats['neither_files'])


In [ ]:
# @title 4. Visualization
# @markdown Display a bar chart of the binding results.
if 'stats' in locals():
    labels = ["Ligand 1 Wins", "Ligand 2 Wins", "Both Bound", "Neither Bound"]
    counts = [stats['lig1_wins'], stats['lig2_wins'], stats['both_bound'], stats['neither_bound']]
    
    fig, ax = plt.subplots()
    bars = ax.bar(labels, counts, color=["blue", "red", "purple", "gray"])
    ax.set_ylabel("Number of Models")
    ax.set_title("AlphaFold Ensemble Competition Results")
    ax.set_ylim(0, max(counts + [1]) * 1.2)
    
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f"{height}",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),  # 3 points vertical offset
            textcoords="offset points",
            ha="center",
            va="bottom",
        )
    
    plt.show()


In [ ]:
# @title 5. Visualization (3D)
# @markdown Renders the highest confidence model of the winning state.
if 'results' not in locals() or len(results) == 0:
    print("No valid models to visualize.")
else:
    # Determine winning state
    winning_state = "Neither"
    winning_count = -1
    winning_results = []
    
    state_counts = [
        ("Ligand 1", stats['lig1_wins'], stats.get('lig1_results', [])),
        ("Ligand 2", stats['lig2_wins'], stats.get('lig2_results', [])),
        ("Both", stats['both_bound'], stats.get('both_results', []))
    ]
    
    for name, count, res_list in state_counts:
        if count > winning_count:
            winning_count = count
            winning_state = name
            winning_results = res_list
    
    if winning_count == 0 or not winning_results:
        print("No dominant binding state found among valid models.")
    else:
        # Find model with highest pLDDT
        best_model = max(winning_results, key=lambda x: x["mean_plddt"])
        best_pdb = os.path.join(ACTUAL_OUTPUT_DIR, best_model["pdb_file"])
        
        chain_lig1 = best_model.get("chain_lig1", "B")
        chain_lig2 = best_model.get("chain_lig2", "C")
        
        print(f"Visualizing Winning State: {winning_state} Bound")
        print(f"Best Model: {best_model['pdb_file']} (pLDDT: {best_model['mean_plddt']:.1f})")
        print(f"Target (Chain A) = Grey | Ligand 1 (Chain {chain_lig1}) = Blue | Ligand 2 (Chain {chain_lig2}) = Red")
        
        with open(best_pdb, 'r') as f:
            pdb_data = f.read()
            
        view = py3Dmol.view(width=800, height=600)
        view.addModel(pdb_data, 'pdb')
        
        # Target (Chain A)
        view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'lightgray'}})
        # Ligand 1
        view.setStyle({'chain': chain_lig1}, {'cartoon': {'color': 'blue'}, 'stick': {'color': 'blue'}})
        # Ligand 2
        view.setStyle({'chain': chain_lig2}, {'cartoon': {'color': 'red'}, 'stick': {'color': 'red'}})
        
        view.zoomTo()
        view.show()
